# Synthetic ESIS-II Images

This report forward-models a single ESIS-II detector frame from a real solar scene:
the active-region pair AR 12565/12567 observed by SDO/AIA on 2016-07-15 00:00 UT.

The scene is derived in two steps that are too expensive (and too dependency-heavy)
to run at documentation build time, so their results are committed as data files in
``docs/reports/f2/_data/`` and loaded below:

1. **Differential emission measure (DEM)**: the six optically-thin AIA EUV channels
   (94, 131, 171, 193, 211, and 335 Å — the 304 Å channel is excluded as usual)
   were corrected for degradation and exposure, reprojected to a common grid,
   binned to 1.2″ pixels, and inverted with
   [demregpy](https://github.com/alasdairwilson/demregpy) over
   $5.6 \le \log T \le 7.2$ against the SSW `evenorm` temperature
   response (`dem.npz`).
2. **Contribution functions**: $G(n_e, T)$ for every line in the 430–535 Å
   ESIS-II passband was computed from the CHIANTI database via
   [fiasco](https://github.com/wtbarnes/fiasco) at
   $n_e = 10^9\,\mathrm{cm}^{-3}$ with Feldman (1992) extended coronal
   abundances — the same abundance set assumed by the AIA temperature response.
   The 21 lines contributing more than 0.5% of the DEM-weighted photon flux
   (94% of the band total) are committed in `gofnt.ecsv`.

Everything downstream of those files — line intensity maps, the field-stop mask,
and the projection onto the detector through the linearized optical model — runs
in this notebook using only ``esis`` and ``optika``.

In [ ]:
import pathlib
import warnings

import astropy.units as u
import matplotlib.pyplot as plt
import named_arrays as na
import numpy as np
from astropy.table import Table

import esis
import optika

directory_data = pathlib.Path("_data")

## Contribution functions

Each row of `gofnt.ecsv` is one spectral line, with its CHIANTI
contribution function sampled at the centers of the DEM temperature bins.
The two ESIS-II target lines, Ne VII 465.2 Å ($\log T \approx 5.7$) and
Si XII 499.4 Å ($\log T \approx 6.3$), bracket the passband in temperature,
and the contaminant lines (Mg VII–IX, Ca IX, Fe XIII–XV, and the Si XII 520.7 Å
doublet partner) fill in between.

In [ ]:
gofnt = Table.read(directory_data / "gofnt.ecsv")
logt = np.array(gofnt.meta["logt"])
gofnt

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
norm = plt.Normalize(430, 535)
for row in gofnt:
    g = row["gofnt"]
    color = plt.cm.viridis(norm(row["wavelength"]))
    lw = 2 if row["ion"] in ("Ne VII", "Si XII") else 0.8
    label = f'{row["ion"]} {row["wavelength"]:.1f} Å' if lw == 2 else None
    ax.plot(logt, g, color=color, lw=lw, label=label)
ax.set_yscale("log")
ax.set_ylim(1e-28, 1e-23)
ax.set_xlabel(r"$\log_{10}(T / \mathrm{K})$")
ax.set_ylabel(r"$G(n_e, T)$ (erg cm$^3$ / s)")
ax.legend()
fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=ax,
    label="wavelength (Å)",
);

## Line intensity maps

The DEM cube gives the emission measure distribution along each line of sight.
The intensity of each line is

$$
I = \frac{1}{4 \pi} \frac{n_H}{n_e} \int G(n_e, T) \, \mathrm{DEM}(T) \, dT,
$$

with $n_H / n_e \approx 0.83$, converted to photon units with the photon
energy of each line.

In [ ]:
dem_file = np.load(directory_data / "dem.npz")
dem = dem_file["dem"]  # (y, x, T) in cm^-5 / K
x_vertices = dem_file["x_vertices_arcsec"] * u.arcsec
y_vertices = dem_file["y_vertices_arcsec"] * u.arcsec
t_edges_log = dem_file["t_edges_log"]
dT = np.diff(10.0 ** t_edges_log)  # K

ratio_nh_ne = 0.83
em = dem * dT  # (y, x, T) in cm^-5

g_matrix = np.stack([row["gofnt"] for row in gofnt])  # (line, T) erg cm^3 / s
intensity = ratio_nh_ne / (4 * np.pi) * np.tensordot(em, g_matrix, axes=([2], [1]))
# (y, x, line) in erg / cm^2 / s / sr

import astropy.constants
energy_photon = (
    astropy.constants.h * astropy.constants.c / gofnt["wavelength"].quantity
).to_value(u.erg)
intensity_photon = intensity / energy_photon  # (y, x, line) in ph / cm^2 / s / sr

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
extent = [
    x_vertices[0].value, x_vertices[-1].value,
    y_vertices[0].value, y_vertices[-1].value,
]
for ax, name, wavelength in zip(axs, ["Ne VII", "Si XII"], [465.221, 499.406]):
    i = np.argmin(np.abs(gofnt["wavelength"] - wavelength))
    ax.imshow(
        np.sqrt(intensity_photon[..., i]),
        origin="lower",
        extent=extent,
        cmap="afmhot",
    )
    ax.set_title(f"{name} {wavelength:.1f} Å")
    ax.set_xlabel("helioprojective x offset (arcsec)")
axs[0].set_ylabel("helioprojective y offset (arcsec)");

## The instrument and the field of view

The forward model is a single channel of the current ESIS-II design.
Replacing the sensor material with an idealized one-electron-per-photon model and
setting a one-second exposure makes the output image read directly in
photons / s / pixel.

The scene must also be masked by the field stop.  The instrument is rolled
−22.5° relative to the sky while the grating and sensor sit at an azimuth of
+22.5°: the two rotations cancel in the sky-to-detector mapping, but the
field-stop octagon (vertex-on-axis in the instrument frame) therefore appears
rotated by 22.5° in sky coordinates — a *flat-top* octagon with edge normals
along multiples of 45°.

In [ ]:
instrument = esis.flights.f2.optics.design_single(num_distribution=0)
system = instrument.system
system.sensor.material = optika.sensors.materials.IdealSensorMaterial()
system.sensor.timedelta_exposure = 1 * u.s

radius_fs = instrument.field_stop.radius_clear
z_fs = instrument.field_stop.transformation.translation.vector.z.ndarray
radius_angular = np.arctan2(radius_fs, np.abs(z_fs)).to(u.arcsec)

x_centers = 0.5 * (x_vertices[:-1] + x_vertices[1:])
y_centers = 0.5 * (y_vertices[:-1] + y_vertices[1:])
x_grid, y_grid = np.meshgrid(x_centers, y_centers)

apothem = radius_angular * np.cos(np.pi / 8)
mask = np.ones(x_grid.shape, dtype=bool)
for k in range(8):
    angle = k * np.pi / 4
    mask &= x_grid * np.cos(angle) + y_grid * np.sin(angle) <= apothem

intensity_photon = intensity_photon * mask[..., np.newaxis]
radius_angular

## Projection onto the detector

Each line is imaged through a linearized version of the optical system
(`optika.systems.SequentialSystem.linearize`), which fits the system's
distortion, vignetting, and effective area and then images the scene by
conservative regridding — the same forward operator used by the Level-4
inversion, and free of the Monte-Carlo sampling noise of a ray trace.

Two practical notes:

* the polynomial fits need several distinct wavelength samples, so each line is
  linearized on a five-point grid spanning ±1 Å;
* lines whose image falls entirely outside the detector (here Mg VIII 430.5 Å,
  at the blue edge of the passband) produce a singular fit and are skipped.

In [ ]:
frame = np.zeros((2048, 1040))

position = na.Cartesian2dVectorArray(
    x=na.ScalarArray(x_vertices, axes="scene_x"),
    y=na.ScalarArray(y_vertices, axes="scene_y"),
)

for i, row in enumerate(gofnt):
    wavelength_line = row["wavelength"]

    wavelength_fit = na.linspace(
        wavelength_line - 1,
        wavelength_line + 1,
        axis="wavelength",
        num=5,
    ) * u.AA
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        linear = system.linearize(wavelength=wavelength_fit, degree=2)

    wavelength = na.ScalarArray(
        u.Quantity([wavelength_line - 0.5, wavelength_line + 0.5]) * u.AA,
        axes="wavelength",
    )
    radiance = na.ScalarArray(
        intensity_photon[..., i] * u.ph / u.cm**2 / u.s / u.sr,
        axes=("scene_y", "scene_x"),
    )
    scene = na.FunctionArray(
        inputs=na.SpectralPositionalVectorArray(
            wavelength=wavelength,
            position=position,
        ),
        outputs=radiance / np.diff(wavelength, axis="wavelength"),
    )

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            images = linear.image(
                scene=scene,
                axis_wavelength="wavelength",
                axis_field=("scene_x", "scene_y"),
                noise=False,
            )
    except np.linalg.LinAlgError:
        print(f"{row['ion']} {wavelength_line:.2f} Å: image off detector, skipped")
        continue

    result = images.outputs.to_value(u.electron)  # = photons / s / pixel
    if result.axes != ("detector_x", "detector_y"):
        result = result.transpose(("detector_x", "detector_y"))
    frame += result.ndarray

## The synthetic frame

The Ne VII and Si XII images of the octagonal field of view land side by side on
the 2048 × 1040 active area, overlapping in the middle.  The Ne VII octagon
(left) shows the sharp 0.5 MK fan loops and moss of the two active regions; the
Si XII octagon (right) shows the smoother 2 MK core emission.  The faint
octagons at the far left are the Mg VII–IX complex at 431–448 Å.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.2), constrained_layout=True)
img = ax.imshow(
    np.sqrt(np.clip(frame.T, 0, None)),
    origin="lower",
    cmap="afmhot",
    aspect="equal",
)
ax.set_xticks([])
ax.set_yticks([])
colorbar = fig.colorbar(img, ax=ax, orientation="horizontal", shrink=0.8, pad=0.02)
colorbar.set_label(r"$\sqrt{\mathrm{intensity}}$  (ph / s / pix)$^{1/2}$");